# E7 · Do knowledge-graph cohort signals add information? (live session)

PLAN §9, E7. Baseline: no KG. Stream C owns it. Three parts:

1. **Signal check**: churn hazard of KG-flagged members, before and after the gym events (reproduces Stream A's `results/a5_event_lift.csv`).
2. **Ranking** (main result): on the held-out test members, does adding the KG flags to the churn model's ranking surface members whose *true* post-event hazard is higher?
3. **Rule-level value**: what the rule base does with the KG facts versus without them.

Plan value under the budget (the optimiser) waits on Stream B's optimiser (`docs/HANDOFFS.md` #9); it is not measured here.

Everything is imported from `anchorai`. It reads the built data (`uv run anchorai data build`) and the **cached** KG facts
(`uv run anchorai kg load`); it never connects to Neo4j.

**Caveats (say them wherever these numbers are quoted).**

- The events, the flagged cohorts and the event effect sizes all come from our **synthetic scenario layer**
  (`docs/SCENARIO.md` "Gym events"). The churn model is trained on the Kaggle snapshot, which has no events, so it cannot see
  them by construction. E7 therefore shows that the KG carries information the model lacks *when events of this kind
  exist*, not that real gyms have such events or that our effect sizes are right.
- The label is the scenario's **true post-event hazard** (`true_churn.event_churn`), not `members.churn`. The Kaggle
  label has no events in it, so it cannot reward a signal about them (on it the flags show lift below 1, `results/c3_signal_churn.csv`).
- The ranking compares against a hazard we generated from features plus the event shift. A ranking by `base_churn` would
  be strong too; the question is only whether the flags add to the model.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anchorai.config import get_settings
from anchorai.discovery.scores import load_features, load_scores
from anchorai.knowledge.signals import kg_facts_path, load_kg_facts
from anchorai.reasoning.engine import load_rule_files, run
from anchorai.reasoning.facts import build_facts
from anchorai.scenario.tables import read_tables

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
FIG, RES = Path("figures"), Path("results")
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)

SEED = get_settings().anchorai_seed
tables = read_tables()
members = tables["members"].set_index("member_id")
scores = load_scores().set_index("member_id")
features = load_features()
kg = load_kg_facts()
assert kg is not None, "run `uv run anchorai kg load` first (cached KG facts missing or older than the data build)"
truth = tables["true_churn"].set_index("member_id")
rules = load_rule_files()

print("seed", SEED, "| KG facts file", kg_facts_path().name, "| members", len(members), "| with a KG row", len(kg))
print("split:", members["split"].value_counts().to_dict())
print("risk bands:", scores["risk_band"].value_counts().to_dict())
print("truth columns:", truth.columns.tolist())

## 1. Signal check

Join the cached KG facts, the churn model's scores and the scenario truth. Two ways to measure a flag's lift:

- **within-member** = mean `event_churn` ÷ mean `base_churn` over the flagged members. This is the fair measure: the same
  people with and without the event.
- **raw** = mean `event_churn` of flagged members ÷ mean over all members. This mixes the event with *who attends classes*
  (class-goers are lower-risk to begin with), so it understates the effect.

**Look for:** the within-member lifts matching Stream A's 1.80 (lost instructor) and 1.32 (cancelled class). `n` should
match too (186 and 170); different numbers mean the data or the KG facts are out of step.

In [ ]:
m = (
    members[["split"]]
    .join(scores[["churn_prob", "risk_band"]])
    .join(truth)
    .join(kg[["lost_anchor_instructor", "class_cancelled"]])
)
for col in ("lost_anchor_instructor", "class_cancelled"):
    m[col] = m[col].fillna(False).astype(bool)

all_event, all_base = m["event_churn"].mean(), m["base_churn"].mean()
rows = []
for name, flag in [("lost_anchor_instructor", m["lost_anchor_instructor"]), ("class_cancelled", m["class_cancelled"])]:
    f = m[flag]
    rows.append({
        "signal": name,
        "n_flagged": len(f),
        "base_churn": f["base_churn"].mean(),
        "event_churn": f["event_churn"].mean(),
        "within_member_lift": f["event_churn"].mean() / f["base_churn"].mean(),
        "raw_lift_vs_all": f["event_churn"].mean() / all_event,
        "mean_churn_prob_model": f["churn_prob"].mean(),
    })
signal = pd.DataFrame(rows).set_index("signal")
signal.round(3).to_csv(RES / "e7_signal_lift.csv")

a5 = pd.read_csv(RES / "a5_event_lift.csv")
a5["signal"] = ["lost_anchor_instructor", "class_cancelled"]
side = signal[["n_flagged", "within_member_lift", "raw_lift_vs_all"]].join(
    a5.set_index("signal")[["members", "within_rise", "lift_vs_all"]].rename(
        columns={"members": "a5_n", "within_rise": "a5_within", "lift_vs_all": "a5_raw"}
    )
)
print(f"all members: base {all_base:.3f}, event {all_event:.3f}")
side.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.2))
x = np.arange(len(signal))
ax.bar(x - 0.2, signal["base_churn"], 0.4, label="before event (base)", color="#8fa8c4")
ax.bar(x + 0.2, signal["event_churn"], 0.4, label="after event", color="#c0504d")
ax.axhline(all_event, ls=":", c="grey", lw=1)
ax.text(1.45, all_event, "all members", va="bottom", ha="right", fontsize=8, color="grey")
ax.set_xticks(x, [f"{s}\n(n={n})" for s, n in zip(signal.index, signal["n_flagged"])])
ax.set_ylabel("mean no-contact churn hazard")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "e7_signal_lift.png")

## 2. Ranking: does the KG add to the churn model?

**Setup.** Test split only (`members.split == "test"`); the churn model was trained on the train split, so its test scores are
out-of-sample. A retention team has a budget of `k` contacts, so what matters is who is at the top of the list.

**Baseline (no KG).** Rank by `churn_prob`.

**With KG.** Add a fixed bonus to `churn_prob` for members with `lost_anchor_instructor` or `class_cancelled`, then rank.
Why this combination:

- It is transparent: a gym manager can read "flagged members count as `b` more at risk".
- It does **not** use the documented event multipliers (1.80 / 1.32) or any scenario truth. Adjusting `churn_prob` by the
  known hazard ratio would be circular: it would hand the ranking the answer it is scored against. The bonus is a single
  number a manager could choose without the simulator.
- Because the bonus is a choice, we show a **sensitivity** over three sizes (0.05, 0.10, 0.20) and a rank-only variant
  (`in-band`): flagged members move ahead of unflagged members *within their risk band* and bands never reorder.

**Scores against truth.** Target is `event_churn` (the true post-event hazard).

- **Spearman** between the ranking score and `event_churn` over all test members.
- **Hazard-weighted precision@k**: the mean `event_churn` of the top `k` members (1.0 = everyone in the list would churn
  without contact). k = 50, 100, 200. An *oracle* ranking by `event_churn` itself is shown as a ceiling.

**Uncertainty.** 1,000 bootstrap resamples of the test members (seeded from `ANCHORAI_SEED`); ranking and scoring are redone
inside each resample, and we report the 95% interval of the **difference** to baseline.

In [ ]:
test = m[m["split"] == "test"].copy()
test["flag"] = test["lost_anchor_instructor"] | test["class_cancelled"]
BONUSES = (0.05, 0.10, 0.20)
KS = (50, 100, 200)
print(f"{len(test)} test members, {int(test['flag'].sum())} KG-flagged "
      f"({int(test['lost_anchor_instructor'].sum())} lost instructor, {int(test['class_cancelled'].sum())} cancelled class)")

band_rank = test["risk_band"].map({"high": 2, "medium": 1, "low": 0})
rankers = {"baseline (churn_prob)": test["churn_prob"].to_numpy()}
for b in BONUSES:
    rankers[f"KG bonus +{b:.2f}"] = (test["churn_prob"] + b * test["flag"]).to_numpy()
# In-band: band first, then flag, then churn_prob (encoded as one sortable number).
rankers["KG in-band"] = (band_rank * 4 + test["flag"] * 2 + test["churn_prob"]).to_numpy()
rankers["oracle (event_churn)"] = test["event_churn"].to_numpy()
y = test["event_churn"].to_numpy()


def spearman(score, y):
    return pd.Series(score).rank().corr(pd.Series(y).rank())


def prec_at(score, y, k):
    # Stable tie-break on position, so equal scores don't depend on library internals.
    top = np.argsort(-score, kind="stable")[:k]
    return y[top].mean()


def metrics(score, y):
    return {"spearman": spearman(score, y), **{f"p@{k}": prec_at(score, y, k) for k in KS}}


point = pd.DataFrame({name: metrics(s, y) for name, s in rankers.items()}).T
point.round(4)

In [ ]:
rng = np.random.default_rng(SEED)
B = 1000
names = list(rankers)
boot = {n: [] for n in names}
for _ in range(B):
    idx = rng.integers(0, len(test), len(test))
    for n in names:
        boot[n].append(metrics(rankers[n][idx], y[idx]))
boot = {n: pd.DataFrame(v) for n, v in boot.items()}

rows = []
for n in names:
    row = {"ranking": n, **{c: point.loc[n, c] for c in point.columns}}
    if n != names[0]:
        for c in point.columns:
            d = boot[n][c] - boot[names[0]][c]
            row[f"{c}_diff"] = point.loc[n, c] - point.loc[names[0], c]
            row[f"{c}_lo"], row[f"{c}_hi"] = d.quantile([0.025, 0.975])
    rows.append(row)
ranking = pd.DataFrame(rows).set_index("ranking")
ranking.round(4).to_csv(RES / "e7_ranking.csv")
ranking.filter(regex="diff|lo|hi").dropna().round(4)

**Look for:** a difference whose 95% interval excludes 0 (both `_lo` and `_hi` on the same side). Compare the three
bonus sizes and `KG in-band`: if only one setting helps, the result depends on a tuned knob and should be reported that way.
The oracle row bounds what any ranking could reach. Only about 8% of test members are flagged, so most of the top `k` is
unaffected by any bonus and the differences are small; judge them by the intervals, not the point estimates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ordered = [n for n in names if not n.startswith("oracle")]
colors = ["#8fa8c4"] + ["#c0504d"] * (len(ordered) - 1)

ax = axes[0]
for i, k in enumerate(KS):
    col = f"p@{k}"
    vals = ranking.loc[ordered, col]
    ax.bar(np.arange(len(ordered)) * 4 + i, vals, 0.9, color=colors, alpha=0.5 + 0.2 * i)
ax.set_xticks(np.arange(len(ordered)) * 4 + 1, [n.replace(" (", "\n(") for n in ordered], fontsize=7)
ax.set_ylabel("mean true hazard of top k  (bars: k = 50, 100, 200)")
ax.set_ylim(ranking.loc[ordered, [f"p@{k}" for k in KS]].min().min() * 0.9, None)

ax = axes[1]
diff = ranking.loc[ordered[1:]]
for i, k in enumerate(KS):
    col = f"p@{k}"
    ax.errorbar(np.arange(len(diff)) + (i - 1) * 0.2, diff[f"{col}_diff"],
                yerr=[diff[f"{col}_diff"] - diff[f"{col}_lo"], diff[f"{col}_hi"] - diff[f"{col}_diff"]],
                fmt="o", capsize=3, label=f"k={k}")
ax.axhline(0, c="grey", lw=1)
ax.set_xticks(np.arange(len(diff)), [n.replace(" (", "\n(") for n in diff.index], fontsize=7)
ax.set_ylabel("precision@k minus baseline (95% bootstrap CI)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "e7_ranking.png")

## 3. Rule-level value: with versus without the KG facts

Run the shipped rule base over **all** members twice. Facts are built exactly as the API builds them
(`build_facts`), once with the cached KG facts and once with `kg=None`, in which case the KG facts are simply missing and
rules on them cannot match (D07 `lost_anchor_instructor`, D08 `class_cancelled`, and whatever playbooks use the
`routine_disrupted` fact they assert).

This is the rule layer only: which members get an allowed offer. How much the extra offers are worth under the weekly
budget, staff minutes and inventory needs Stream B's optimiser (`docs/HANDOFFS.md` #9). As a proxy we report the
**event-driven excess hazard** `event_churn − base_churn` of members reached only because of the KG: the churn the events
added to them, which is what a timely offer could at most address.

In [ ]:
interventions = tables["interventions"]
facts_with = build_facts(scores, features, interventions, kg)
facts_without = build_facts(scores, features, interventions, None)
out_with = {mid: run(rules, f, mid) for mid, f in facts_with.items()}
out_without = {mid: run(rules, f, mid) for mid, f in facts_without.items()}

per = pd.DataFrame({
    "offers_with": {k: tuple(p.offer_id for p in o.allowed) for k, o in out_with.items()},
    "offers_without": {k: tuple(p.offer_id for p in o.allowed) for k, o in out_without.items()},
    "rules_with": {k: {f.rule_id for f in o.firings} for k, o in out_with.items()},
    "rules_without": {k: {f.rule_id for f in o.firings} for k, o in out_without.items()},
})
per["n_with"] = per["offers_with"].map(len)
per["n_without"] = per["offers_without"].map(len)
per["gains"] = (per["n_with"] > 0) & (per["n_without"] == 0)
per["loses"] = (per["n_with"] == 0) & (per["n_without"] > 0)
per["new_offers"] = [sorted(set(a) - set(b)) for a, b in zip(per["offers_with"], per["offers_without"])]
per["dropped_offers"] = [sorted(set(b) - set(a)) for a, b in zip(per["offers_with"], per["offers_without"])]
per["changed"] = per["offers_with"] != per["offers_without"]
per["new_rules"] = [sorted(a - b) for a, b in zip(per["rules_with"], per["rules_without"])]
per = per.join(truth[["base_churn", "event_churn"]])
per["excess"] = per["event_churn"] - per["base_churn"]

summary = pd.Series({
    "members": len(per),
    "with an allowed offer, with KG": int((per["n_with"] > 0).sum()),
    "with an allowed offer, without KG": int((per["n_without"] > 0).sum()),
    "gain an allowed offer (none -> some)": int(per["gains"].sum()),
    "lose all allowed offers (some -> none)": int(per["loses"].sum()),
    "offer list changes at all": int(per["changed"].sum()),
    "members reached only with KG: sum excess hazard": per.loc[per["gains"], "excess"].sum(),
    "members reached only with KG: mean excess hazard": per.loc[per["gains"], "excess"].mean(),
    "members whose list changes: sum excess hazard": per.loc[per["changed"], "excess"].sum(),
    "all members: sum excess hazard": per["excess"].sum(),
})
summary.round(3).to_frame("value")

In [ ]:
changed = per[per["changed"]]
by_rule = (
    changed.explode("new_rules").dropna(subset=["new_rules"]).groupby("new_rules")
    .agg(members=("excess", "size"), gain_offer=("gains", "sum"), sum_excess=("excess", "sum"))
)
by_offer = pd.Series([o for offers in changed["new_offers"] for o in offers]).value_counts().rename("members_newly_offered")
dropped = pd.Series([o for offers in changed["dropped_offers"] for o in offers]).value_counts().rename("members_no_longer_offered")

table = by_rule.assign(kind="rule firing only with KG").rename_axis("item").reset_index()
table = pd.concat([
    pd.DataFrame({"item": summary.index, "value": summary.values, "kind": "summary"}),
    table.rename(columns={"members": "value"})[["item", "value", "kind", "gain_offer", "sum_excess"]],
    by_offer.rename_axis("item").reset_index().rename(columns={"members_newly_offered": "value"}).assign(kind="offer newly allowed"),
    dropped.rename_axis("item").reset_index().rename(columns={"members_no_longer_offered": "value"}).assign(kind="offer no longer allowed"),
], ignore_index=True)
table.round(3).to_csv(RES / "e7_rules_with_without_kg.csv", index=False)
print("rules that fire only because of the KG facts:")
display(by_rule.round(3))
print("offers newly allowed:", by_offer.to_dict(), "| no longer allowed:", dropped.to_dict())

**Look for:** (a) which rules appear in `new_rules` besides D07/D08 (the playbooks fed by `routine_disrupted`);
(b) members who *lose* an offer with the KG: that means a KG fact turned on a guardrail or changed priorities, so check it
is intended; (c) how many KG-flagged members are already reached without the KG (the churn model already puts them
in a high band), since those add nothing at the rule layer.

In [ ]:
flagged = per.join(m[["lost_anchor_instructor", "class_cancelled", "risk_band"]]).query("lost_anchor_instructor or class_cancelled")
pd.crosstab(flagged["risk_band"], flagged["gains"].map({True: "reached only with KG", False: "not newly reached"}), margins=True)

## Findings

*(fill in after running)*

- **Signal.** Within-member lift for lost instructor / cancelled class: _x_ / _y_ (A's: 1.80 / 1.32). Raw lift vs all members: _x_ / _y_; why it is lower.
- **Ranking.** Spearman: baseline _x_, with KG (best setting) _y_; difference _d_ [95% CI _lo_, _hi_]. Precision@50/100/200: _..._ . Does it depend on the bonus size? Does `in-band` help?
- **Rule layer.** _n_ members gain an allowed offer only with the KG, _n_ lose one; rules responsible: _..._ ; excess hazard of the newly reached members: _x_.
- **Limits.** Synthetic events and effect sizes; churn model blind to events by construction; plan value under the budget is pending Stream B's optimiser.
- **Verdict for the report.** _(does the KG add information, how much, at what cost)_